# 🚀 Huấn Luyện Toàn Bộ 6 Cấu Hình Ablation Study D2SA Trên NVIDIA A100 (40GB VRAM)

**Dự án:** Amodal Shape Prediction (Swin-UNet + SAM 2.1) — Mở rộng sang tập dữ liệu **MVTec D2SA**  
**Môi trường:** Google Colab Pro / Pro+ với GPU **NVIDIA A100 (40GB VRAM)**  

### ⚡ Các Tính Năng & Tối Ưu Hóa Nổi Bật:
- 📦 **Tự Động Nhận Diện & Giải Nén Siêu Tốc `.tar.xz`**: Hỗ trợ trực tiếp 2 file nén đã tải lên Google Drive tại `/content/drive/MyDrive/amodal_shape_prediction/data/D2SA`:
  - `d2s_amodal_annotations_v1.tar.xz` (~15.7 MB)
  - `d2s_amodal_images_v1.tar.xz` (~4.3 GB, 22,562 ảnh)
  - Tận dụng giải nén đa luồng `tar -I "xz -T0"` trực tiếp vào ổ SSD NVMe của Colab trong ~1–2 phút. Tránh 100% tình trạng nghẽn I/O khi đọc 22,562 file ảnh qua mạng Google Drive FUSE.
- 🔄 **Lưu Trực Tiếp Vào Google Drive**: Checkpoints (`.pth`), Text Logs (`.log`), Metrics (`.jsonl`), và Báo cáo (`.json`) được ghi thẳng vào Google Drive sau mỗi epoch với cơ chế `flush` + `fsync` nguyên tử (chống mất mát dữ liệu khi Colab ngắt kết nối).
- ⚡ **Tối Ưu Hóa A100 40GB**: Huấn luyện với `batch_size=16`, `accumulation_steps=1`, `num_workers=8`. VRAM tiêu thụ ~12-13 GB (chỉ chiếm ~30% VRAM của A100 40GB, loại bỏ 100% rủi ro OOM). Kích hoạt Tensor Cores TF32 và cuDNN Benchmark.
- 📊 **Tự Động Tổng Hợp Table V & Biểu Đồ Trực Quan**: Tự động xuất bảng so sánh 6 cấu hình (mIoU, Dice, Precision, Recall, Invisible mIoU) chuẩn Markdown, Pandas DataFrame và vẽ biểu đồ so sánh trực quan.

---

### 📌 Bước 1: Kiểm Tra GPU & Thông Số Phần Cứng
Đảm bảo bạn đã chọn Runtime: **Runtime -> Change runtime type -> A100 GPU -> High-RAM**.

In [ ]:
!nvidia-smi

import os
import psutil
import torch

print("\n" + "=" * 65)
print("THÔNG TIN HỆ THỐNG & PHẦN CỨNG COLAB")
print("=" * 65)
print("PyTorch Version :", torch.__version__)
print("CUDA Available  :", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    compute_cap = torch.cuda.get_device_capability(0)
    print(f"GPU Name        : {gpu_name}")
    print(f"Total VRAM      : {vram_gb:.2f} GB")
    print(f"Compute Cap     : {compute_cap[0]}.{compute_cap[1]}")

    # Kích hoạt tăng tốc phần cứng cho NVIDIA Ampere A100
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")
        print("⚡ Đã kích hoạt Tensor Cores TF32 (float32_matmul_precision = 'high')")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Vui lòng vào Runtime -> Change runtime type -> A100 GPU")

# Kiểm tra CPU & RAM
cpu_count = os.cpu_count()
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"vCPUs Available : {cpu_count} cores (Tối ưu cho num_workers=8 & giải nén đa luồng)")
print(f"System RAM      : {ram_gb:.2f} GB")
print("=" * 65)

### 📁 Bước 2: Kết Nối Google Drive Để Lưu Trực Tiếp
Toàn bộ checkpoints và kết quả sẽ được lưu liên tục vào Google Drive để không bị mất khi Colab ngắt kết nối.

In [ ]:
from google.colab import drive
import os
import shutil

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Thiết lập thư mục lưu trữ trên Google Drive của bạn
DRIVE_DIR = "/content/drive/MyDrive/amodal_shape_prediction"
os.makedirs(f"{DRIVE_DIR}/checkpoints/d2sa", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/logs/d2sa", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/results/d2sa", exist_ok=True)

# 3. Kiểm tra dung lượng còn trống trên Drive
try:
    total, used, free = shutil.disk_usage("/content/drive/MyDrive")
    print(f"✅ Đã kết nối Google Drive! Thư mục lưu trữ: {DRIVE_DIR}")
    print(f"💾 Dung lượng Drive còn trống: {free / (1024**3):.2f} GB / {total / (1024**3):.2f} GB")
except Exception:
    print(f"✅ Đã kết nối Google Drive! Thư mục lưu trữ: {DRIVE_DIR}")

### 📥 Bước 3: Chuẩn Bị Mã Nguồn & Cài Đặt Thư Viện
Tự động đồng bộ mã nguồn từ Google Drive (nếu đã tải lên) hoặc clone từ GitHub branch `feat/colab-a100`.  
> ⚡ **Lưu ý tối ưu:** Khi đồng bộ mã nguồn, tự động loại trừ thư mục `data/` và `checkpoints/` để sao chép mã nguồn siêu tốc (~1–2 giây) mà không bị nghẽn I/O qua Drive FUSE.

In [ ]:
import os

DRIVE_REPO = "/content/drive/MyDrive/amodal_shape_prediction"
LOCAL_REPO = "/content/amodal_shape_prediction"
BRANCH = "new_framework"

os.makedirs(LOCAL_REPO, exist_ok=True)

# 1. Nếu đã có repo trên Google Drive -> đồng bộ mã nguồn sang SSD Colab (loại trừ data & checkpoints nặng)
if os.path.exists(os.path.join(DRIVE_REPO, "scripts")):
    print("📁 Phát hiện repo có sẵn trên Google Drive! Đồng bộ mã nguồn sang SSD Colab...")
    !rsync -av --exclude 'data' --exclude 'checkpoints' --exclude 'logs' --exclude '.git' "{DRIVE_REPO}/" "{LOCAL_REPO}/" 2>/dev/null || (cd "{DRIVE_REPO}" && cp -r scripts docs assets *.py *.md requirements.txt "{LOCAL_REPO}/" 2>/dev/null)
    %cd {LOCAL_REPO}
# 2. Nếu repo cục bộ đã tồn tại (đã clone từ trước trên Colab):
elif os.path.exists(os.path.join(LOCAL_REPO, ".git")):
    print(f"🔄 Thư mục cục bộ đã tồn tại, đang fetch và checkout branch {BRANCH}...")
    %cd {LOCAL_REPO}
    !git fetch origin
    !git checkout {BRANCH} 2>/dev/null || git checkout main
    !git pull origin {BRANCH} 2>/dev/null || git pull origin main
# 3. Nếu chưa có trên Drive và chưa có cục bộ -> clone trực tiếp từ GitHub:
else:
    print(f"🌐 Chưa có repo, đang clone branch {BRANCH} từ GitHub...")
    !rm -rf {LOCAL_REPO}
    !git clone -b {BRANCH} https://github.com/vothenguyen/amodal_shape_prediction.git {LOCAL_REPO} || git clone https://github.com/vothenguyen/amodal_shape_prediction.git {LOCAL_REPO}
    %cd {LOCAL_REPO}

# Cài đặt các dependencies cần thiết
print("\n📦 Cài đặt thư viện dependencies...")
!pip install -q timm albumentations pycocotools opencv-python tqdm pandas matplotlib
print("✅ Môi trường Python đã sẵn sàng!")

### 📦 Bước 4: Thiết Lập & Tự Động Nhận Diện Dữ Liệu Nén D2SA Trên Drive

> 💡 **Cơ chế nạp dữ liệu siêu tốc cho Colab A100:**
> - Bạn đã đưa 2 file nén lên Google Drive tại: `/content/drive/MyDrive/amodal_shape_prediction/data/D2SA/`:
>   - `d2s_amodal_annotations_v1.tar.xz` (~15.7 MB): Chứa 3 file annotation JSON (`D2S_amodal_training_rot0.json`, `D2S_amodal_augmented.json`, `D2S_amodal_validation.json`).
>   - `d2s_amodal_images_v1.tar.xz` (~4.3 GB): Chứa thư mục `images/` với 22,562 ảnh JPG.
> - **Chiến lược tối ưu hóa:** Đoạn mã bên dưới sẽ đọc trực tiếp 2 file nén từ Drive và **giải nén đa luồng (`tar -I "xz -T0"`) thẳng vào ổ SSD NVMe của Colab** (`/content/amodal_shape_prediction/data/D2SA/`):
>   1. **Tốc độ đọc SSD > 2 GB/s**: Trong quá trình huấn luyện, DataLoader với `num_workers=8` đọc 22,562 ảnh tức thì, giúp GPU A100 luôn chạy ở 100% công suất.
>   2. **Tránh lỗi nghẽn I/O**: Giải nén lên SSD chỉ mất ~1–2 phút một lần duy nhất, tránh hoàn toàn độ trễ đọc file qua Google Drive FUSE.
>   3. **Tự động bỏ qua**: Nếu dữ liệu đã được giải nén sẵn trên SSD Colab, hệ thống sẽ tự động phát hiện và bỏ qua trong 0.1 giây!

In [ ]:
import os
import sys
import time
import glob
import shutil

LOCAL_DATA_DIR = "/content/amodal_shape_prediction/data/D2SA"
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

REQ_JSONS = [
    "D2S_amodal_training_rot0.json",
    "D2S_amodal_augmented.json",
    "D2S_amodal_validation.json"
]
IMG_DIR = os.path.join(LOCAL_DATA_DIR, "images")

# 1. Kiểm tra xem dữ liệu đã sẵn sàng trên SSD Colab chưa
def check_dataset_ready(data_dir):
    has_all_jsons = all(os.path.isfile(os.path.join(data_dir, j)) for j in REQ_JSONS)
    images_path = os.path.join(data_dir, "images")
    has_images = os.path.isdir(images_path) and len(os.listdir(images_path)) >= 20000
    return has_all_jsons, has_images

has_jsons, has_images = check_dataset_ready(LOCAL_DATA_DIR)

if has_jsons and has_images:
    img_count = len(os.listdir(IMG_DIR))
    print("=" * 70)
    print("🎉 DỮ LIỆU D2SA ĐÃ CÓ SẴN VÀ ĐẦY ĐỦ TRÊN SSD COLAB!")
    print(f"📁 Thư mục dữ liệu : {LOCAL_DATA_DIR}")
    print(f"📄 Các file nhãn   : 3/3 JSONs đã sẵn sàng")
    print(f"🖼️ Tổng số ảnh     : {img_count:,} ảnh trong images/")
    print("⚡ Bỏ qua bước giải nén để tiết kiệm thời gian.")
    print("=" * 70)
else:
    print("=" * 70)
    print("📦 BẮT ĐẦU TỰ ĐỘNG PHÁT HIỆN & GIẢI NÉN DỮ LIỆU D2SA TỪ GOOGLE DRIVE")
    print("=" * 70)

    # Danh sách các vị trí tìm kiếm trên Google Drive và cục bộ
    # Ưu tiên vị trí người dùng đã đặt: /content/drive/MyDrive/amodal_shape_prediction/data/D2SA
    SEARCH_DIRS = [
        f"{DRIVE_DIR}/data/D2SA",
        "/content/drive/MyDrive/amodal_shape_prediction/data/D2SA",
        f"{DRIVE_DIR}/data",
        f"{DRIVE_DIR}",
        "/content/drive/MyDrive/D2SA",
        "/content/drive/MyDrive/data/D2SA",
        "/content/drive/MyDrive/data",
        "/content/drive/MyDrive",
        "/content/amodal_shape_prediction/data/D2SA",
        "/content/amodal_shape_prediction/data",
        "/content/data/D2SA",
        "/content"
    ]

    def find_file(filenames_or_patterns):
        for sdir in SEARCH_DIRS:
            if not os.path.exists(sdir):
                continue
            for name in filenames_or_patterns:
                target = os.path.join(sdir, name)
                if os.path.isfile(target):
                    return target
                matches = glob.glob(os.path.join(sdir, name))
                if matches and os.path.isfile(matches[0]):
                    return matches[0]
        return None

    # Tìm file nén Annotations và Images
    ann_archive = find_file([
        "d2s_amodal_annotations_v1.tar.xz",
        "d2s_amodal_annotations.tar.xz",
        "*amodal*annotation*.tar.xz",
        "*annotation*.tar.xz",
        "*annotations*.zip"
    ])

    img_archive = find_file([
        "d2s_amodal_images_v1.tar.xz",
        "d2s_amodal_images.tar.xz",
        "*amodal*image*.tar.xz",
        "*image*.tar.xz",
        "*images*.tar.xz",
        "*images*.zip"
    ])

    combined_archive = find_file([
        "D2SA.tar.xz",
        "D2SA.zip",
        "d2s_amodal.zip",
        "D2SA.tar.gz"
    ])

    # Hàm giải nén tối ưu đa luồng
    def extract_archive(archive_path, dest_dir):
        size_mb = os.path.getsize(archive_path) / (1024 * 1024)
        print(f"\n🚀 Đang giải nén: {os.path.basename(archive_path)} ({size_mb:.1f} MB)")
        print(f"   Từ : {archive_path}")
        print(f"   Đến: {dest_dir}")
        t0 = time.time()

        if archive_path.endswith(".tar.xz") or archive_path.endswith(".txz"):
            # Sử dụng xz -T0 để tận dụng toàn bộ vCPUs giải nén đa luồng
            cmd = f'tar -I "xz -T0" -xf "{archive_path}" -C "{dest_dir}"'
            ret = os.system(cmd)
            if ret != 0:
                print("   ⚠️ Lệnh tar đa luồng gặp lỗi, rơi về lệnh tar tiêu chuẩn...")
                ret = os.system(f'tar -xf "{archive_path}" -C "{dest_dir}"')
            if ret != 0:
                raise RuntimeError(f"Giải nén thất bại: {archive_path}")
        elif archive_path.endswith(".zip"):
            ret = os.system(f'unzip -q -o "{archive_path}" -d "{dest_dir}"')
            if ret != 0:
                raise RuntimeError(f"Giải nén thất bại: {archive_path}")
        elif archive_path.endswith(".tar.gz") or archive_path.endswith(".tgz"):
            ret = os.system(f'tar -xzf "{archive_path}" -C "{dest_dir}"')
            if ret != 0:
                raise RuntimeError(f"Giải nén thất bại: {archive_path}")
        elif archive_path.endswith(".tar"):
            ret = os.system(f'tar -xf "{archive_path}" -C "{dest_dir}"')
            if ret != 0:
                raise RuntimeError(f"Giải nén thất bại: {archive_path}")

        elapsed = time.time() - t0
        print(f"   ✅ Giải nén xong trong {elapsed:.1f} giây!")

    # 2. Giải nén Annotations nếu thiếu
    if not has_jsons:
        if ann_archive:
            print(f"🔍 Tìm thấy file annotations: {ann_archive}")
            extract_archive(ann_archive, LOCAL_DATA_DIR)
        elif combined_archive:
            print(f"🔍 Tìm thấy file nén tổng hợp: {combined_archive}")
            extract_archive(combined_archive, LOCAL_DATA_DIR)
        else:
            print("⚠️ Không tìm thấy file nén annotations (d2s_amodal_annotations_v1.tar.xz)!")

    # 3. Giải nén Images nếu thiếu
    if not has_images:
        if img_archive:
            print(f"🔍 Tìm thấy file ảnh nén: {img_archive}")
            extract_archive(img_archive, LOCAL_DATA_DIR)
        elif combined_archive and not ann_archive:
            extract_archive(combined_archive, LOCAL_DATA_DIR)
        else:
            print("⚠️ Không tìm thấy file nén images (d2s_amodal_images_v1.tar.xz)!")

    # 4. Tự động kiểm tra và di chuyển file nếu archive tạo thư mục con lồng nhau
    for root, dirs, files in os.walk(LOCAL_DATA_DIR):
        if root == LOCAL_DATA_DIR:
            continue
        for req_f in REQ_JSONS:
            if req_f in files and not os.path.exists(os.path.join(LOCAL_DATA_DIR, req_f)):
                src_path = os.path.join(root, req_f)
                dst_path = os.path.join(LOCAL_DATA_DIR, req_f)
                print(f"📁 Di chuyển {req_f} về {LOCAL_DATA_DIR}...")
                shutil.move(src_path, dst_path)
        if os.path.basename(root) == "images" and root != IMG_DIR:
            if not os.path.exists(IMG_DIR):
                print(f"📁 Di chuyển thư mục images về {IMG_DIR}...")
                shutil.move(root, IMG_DIR)

    # 5. Xác nhận tính sẵn sàng cuối cùng
    has_jsons, has_images = check_dataset_ready(LOCAL_DATA_DIR)
    if has_jsons and has_images:
        print("\n" + "=" * 70)
        print("✅ THIẾT LẬP DỮ LIỆU D2SA THÀNH CÔNG VÀ SẴN SÀNG HUẤN LUYỆN!")
        print(f"📁 Thư mục  : {LOCAL_DATA_DIR}")
        print(f"🖼️ Tổng ảnh : {len(os.listdir(IMG_DIR)):,} ảnh")
        print("=" * 70)
    else:
        print("\n" + "❌" * 35)
        print("LỖI: CHƯA THỂ CHUẨN BỊ ĐẦY ĐỦ DỮ LIỆU D2SA!")
        if not has_jsons:
            print("  - Thiếu ít nhất 1 file JSON trong danh sách:", REQ_JSONS)
        if not has_images:
            curr_count = len(os.listdir(IMG_DIR)) if os.path.isdir(IMG_DIR) else 0
            print(f"  - Thiếu thư mục images/ hoặc số ảnh quá ít ({curr_count}/22,562 ảnh)")
        print("\n💡 VUI LÒNG KIỂM TRA:")
        print("  Đảm bảo 2 file sau đã tải lên đúng Google Drive:")
        print(f"    - {DRIVE_DIR}/data/D2SA/d2s_amodal_annotations_v1.tar.xz")
        print(f"    - {DRIVE_DIR}/data/D2SA/d2s_amodal_images_v1.tar.xz")
        print("❌" * 35)

### 🔍 Bước 5: Kiểm Tra Tính Toàn Vẹn Dataset
Xác minh toàn bộ 3 file JSON annotations và 22,562 ảnh JPG.

In [ ]:
!python scripts/verify_d2sa_dataset.py --data-dir /content/amodal_shape_prediction/data/D2SA

### 🧪 Bước 6 (Tùy chọn): Smoke Test Nhanh (1 Epoch trên 50 Mẫu)
Chạy thử nhanh cấu hình chính (Row 4) trên 50 mẫu để kiểm tra toàn diện luồng nạp dữ liệu, forward pass, backward pass, ghi log Drive và tính metrics (chỉ mất ~30 giây).

In [ ]:
!python scripts/run_ablation_d2sa.py \
    --rows 4 \
    --mode all \
    --epochs 1 \
    --subset-size 50 \
    --batch-size 16 \
    --accumulation-steps 1 \
    --num-workers 4 \
    --device cuda \
    --drive-dir {DRIVE_DIR}

### 🏆 Bước 7: CHẠY 1 LẦN ĐẦY ĐỦ TOÀN BỘ 6 CẤU HÌNH (TABLE V) TRÊN A100 (40GB VRAM)

Lệnh dưới đây sẽ tự động huấn luyện và đánh giá tuần tự toàn bộ **6 cấu hình Ablation (Row 1 -> Row 6)**:
- **Epochs:** 30 epochs mỗi cấu hình.
- **Batch size:** 16, **Accumulation steps:** 1 (Effective batch = 16, tốn ~12.5GB / 40GB VRAM, hoàn toàn an toàn).
- **Num workers:** 8 (Tối đa băng thông CPU nạp ảnh qua NVMe SSD).
- **Lưu trữ:** Mọi checkpoint, log và file metrics được đồng bộ tức thì vào Google Drive.
- **Đánh giá:** Tự động đánh giá trên toàn bộ tập Validation (15,654 mẫu) và tổng hợp Table V.

In [ ]:
# =============================================================================
# TUỲ CHỌN 1 (KHUYÊN DÙNG NẾU MUỐN CHẠY RIÊNG CẤU HÌNH CHÍNH TỪ ĐẦU - ROW 4 MỚI):
# Huấn luyện mới toanh Row 4 (Main Config: 5ch, OccLoss 5x, Edge, Emb 60, No Spatial)
# từ Epoch 1 -> 30, lưu trực tiếp vào Drive:
# =============================================================================
!python scripts/run_ablation_d2sa.py \
    --rows 4 \
    --mode all \
    --epochs 30 \
    --batch-size 16 \
    --accumulation-steps 1 \
    --num-workers 8 \
    --device cuda \
    --drive-dir {DRIVE_DIR}

# -----------------------------------------------------------------------------
# TUỲ CHỌN 2 (NẾU MUỐN CHẠY TỪ ROW 4 ĐẾN ROW 6 MỚI TOANH):
# Bỏ qua Row 1, 2, 3 và chạy tuần tự Row 4 -> Row 5 -> Row 6:
# !python scripts/run_ablation_d2sa.py \
#     --rows 4 5 6 \
#     --mode all \
#     --epochs 30 \
#     --batch-size 16 \
#     --accumulation-steps 1 \
#     --num-workers 8 \
#     --device cuda \
#     --drive-dir {DRIVE_DIR}

# -----------------------------------------------------------------------------
# TUỲ CHỌN 3 (CHẠY TOÀN BỘ 6 CẤU HÌNH TABLE V TỪ ROW 1 -> ROW 6):
# !python scripts/run_ablation_d2sa.py \
#     --rows 1 2 3 4 5 6 \
#     --mode all \
#     --epochs 30 \
#     --batch-size 16 \
#     --accumulation-steps 1 \
#     --num-workers 8 \
#     --device cuda \
#     --drive-dir {DRIVE_DIR}


### 🔄 Bước 8: Tiếp Tục Huấn Luyện Khi Bị Gián Đoạn (Resume) Hoặc Làm Sạch Checkpoint Cũ

- **Nếu muốn xóa checkpoint cũ của Row 4 để huấn luyện sạch 100% từ đầu:** Chạy lệnh xóa thư mục checkpoint Row 4 trên Drive.
- **Nếu Colab bị timeout/ngắt kết nối giữa chừng:** Chỉ cần nạp lại Drive và dùng tham số `--resume-epoch` để tiếp tục chính xác từ epoch vừa lưu.

In [ ]:
# =============================================================================
# CÁCH 1: Xóa sạch checkpoint cũ của Row 4 trên Drive nếu muốn train mới 100%:
# =============================================================================
# !rm -rf {DRIVE_DIR}/checkpoints/d2sa/amodal_shape_prediction_main_config
# print("Đã làm sạch checkpoint Row 4 cũ trên Drive!")

# =============================================================================
# CÁCH 2: Resume tiếp tục khi bị ngắt kết nối (VD: Row 4 dừng ở Epoch 12 -> chạy đến 30):
# =============================================================================
# !python scripts/run_ablation_d2sa.py \
#     --rows 4 \
#     --mode all \
#     --epochs 30 \
#     --batch-size 16 \
#     --accumulation-steps 1 \
#     --num-workers 8 \
#     --resume-epoch 12 \
#     --drive-dir {DRIVE_DIR}

# =============================================================================
# CÁCH 3: Chỉ chạy đánh giá (Evaluation) trên tập validation từ checkpoint có sẵn:
# =============================================================================
# !python scripts/run_ablation_d2sa.py \
#     --rows 4 \
#     --mode eval \
#     --device cuda \
#     --drive-dir {DRIVE_DIR}


### 📊 Bước 9: Xem Bảng Tổng Hợp Kết Quả Table V & Biểu Đồ Trực Quan
Hiển thị bảng kết quả so sánh giữa 6 cấu hình (Markdown & Pandas DataFrame) và tự động vẽ biểu đồ so sánh mIoU và Invisible mIoU.

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# 1. Hiển thị bảng tổng hợp Markdown
summary_md_path = f"{DRIVE_DIR}/results/d2sa/table_v_summary.md"
if os.path.exists(summary_md_path):
    print("=" * 80)
    print("📜 BÁO CÁO TABLE V (MARKDOWN)")
    print("=" * 80)
    with open(summary_md_path, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))

# 2. Hiển thị bảng tương tác bằng Pandas DataFrame
summary_json_path = f"{DRIVE_DIR}/results/d2sa/table_v_summary.json"
if os.path.exists(summary_json_path):
    print("\n" + "=" * 80)
    print("📊 BẢNG DỮ LIỆU TỔNG HỢP (DATAFRAME)")
    print("=" * 80)
    with open(summary_json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    display(df)

    # 3. Vẽ biểu đồ so sánh trực quan giữa 6 cấu hình
    try:
        fig, ax = plt.subplots(figsize=(11, 5), dpi=120)
        x = range(len(df))
        width = 0.35

        bar1 = ax.bar([i - width/2 for i in x], df["mIoU"], width, label="mIoU (%)", color="#2b5c8f")
        bar2 = ax.bar([i + width/2 for i in x], df["inv_mIoU"], width, label="Invisible mIoU (%)", color="#e67e22")

        ax.set_xlabel("Cấu hình Ablation Study", fontsize=11, fontweight="bold")
        ax.set_ylabel("Độ chính xác (%)", fontsize=11, fontweight="bold")
        ax.set_title("So Sánh Hiệu Năng 6 Cấu Hình Ablation Study Trên MVTec D2SA (Table V)", fontsize=13, fontweight="bold", pad=12)
        ax.set_xticks(list(x))
        ax.set_xticklabels([f"Row {r}" for r in df["row"]], fontweight="bold")
        ax.legend(frameon=True, loc="upper left")
        ax.set_ylim(0, 100)
        ax.grid(axis='y', linestyle='--', alpha=0.5)

        for bar in bar1:
            h = bar.get_height()
            ax.annotate(f"{h:.1f}%", xy=(bar.get_x() + bar.get_width() / 2, h),
                        xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9, fontweight='bold')

        for bar in bar2:
            h = bar.get_height()
            ax.annotate(f"{h:.1f}%", xy=(bar.get_x() + bar.get_width() / 2, h),
                        xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9, fontweight='bold')

        plt.tight_layout()
        chart_path = f"{DRIVE_DIR}/results/d2sa/ablation_comparison_chart.png"
        plt.savefig(chart_path, dpi=200)
        plt.show()
        print(f"🖼️ Đã lưu biểu đồ tại: {chart_path}")
    except Exception as e_plot:
        print(f"Lưu ý: Không thể vẽ biểu đồ ({e_plot})")
else:
    print(f"Chưa tìm thấy file kết quả JSON tại: {summary_json_path}")